# Export and Publishing

**Part I · Visualization** — Tutorial 19

Export visualizations for sharing and publication. You will learn to:

- Generate self-contained HTML (`export_snapshot`) and embeddable figures
  (`export_figure` + `FigureStyle`).
- Export glTF/GLB (`export_glb`).
- Capture PNG screenshots and MP4 video (`SceneExporter`).
- Preview inline (`display_snapshot`) and standalone (`open_snapshot`).
- Record and export animations (`start_animation_recording` + `AnimStyle`).


## Setup


In [1]:
import math

from pytanga.geometry import Direction, Line, Point, Sphere
from pytanga.viz import (
    AnimStyle, FigureStyle, PointStyle, SphereStyle, Visualizer,
)


## 1. Self-contained HTML — `export_snapshot()`

`export_snapshot(path)` writes a standalone `.html` file (scene data and
renderer modules embedded inline; Three.js still loads from CDN). Path
resolution is forgiving (missing directories created, `~` expanded, missing
extension appended).


In [2]:
viz = Visualizer(title="Export — HTML")
viz.add(Point(2, 0, 0), color="#ff4444", style=PointStyle(size=0.15), label="$P$")
viz.add(Sphere(Point(0, 0, 0), 2.0), style=SphereStyle(wireframe=True), opacity=0.3)

viz.flush()  # resolve any pending dirty state before export
viz.export_snapshot("_output/19_scene.html", overwrite=True)


## 2. Embeddable figures — `export_figure()` + `FigureStyle`

`export_figure()` produces an HTML snippet (`<div>` + `<script type="module">`)
for embedding in a slide deck or page — no `<html>`/`<head>`. Pass `path=None`
to get the snippet back as a string. `FigureStyle` controls `width`, `height`,
`background`, `auto_rotate`, `show_title`, `show_annotation`, `border_radius`,
`responsive`.


In [3]:
viz = Visualizer(title="Sphere Construction")
viz.add(Sphere(Point(0, 0, 0), 2.5), style=SphereStyle(wireframe=True), opacity=0.4, label="$S_1$")
viz.add(Point(0, 0, 0), color="#ffff00", style=PointStyle(size=0.15), label="$O$")

viz.flush()
viz.export_figure(
    "_output/19_figure.html",
    style=FigureStyle(width=800, height=600, background="transparent", auto_rotate=True,
                      border_radius="8px"),
    overwrite=True,
)


## 3. glTF/GLB — `export_glb()`

`export_glb(path)` writes a glTF 2.0 binary `.glb` (the glTF/GLB entry point —
there is no `export_gltf`). Only entities are exported; overlay layers (labels,
title, annotation) are excluded.


In [4]:
viz = Visualizer(title="Export — GLB")
viz.add(Sphere(Point(0, 0, 0), 2.0), opacity=0.4)
viz.add(Line(origin=Point(0, 0, 0), direction=Direction(1, 0, 0)), color="#44ff44")

viz.flush()
viz.export_glb("_output/19_scene.glb", overwrite=True)


## 4. Screenshots and video — `SceneExporter`

Screenshots and video capture the **live browser viewport**, so the server must
be running (and a browser connected). These currently live on `SceneExporter`
(deprecated):

- `SceneExporter(viz).screenshot(path, width=..., height=...)` — full-viewport
  PNG (including overlays).
- `start_capture(...)` / `capture_frame()` / `finish_capture(video_path, fps)`
  — sequenced PNGs stitched into MP4 with `ffmpeg` (on `PATH`).


In [5]:
from pytanga.viz import SceneExporter

# (requires a running viewer + connected browser)
# viz.show()
# exporter = SceneExporter(viz)
# exporter.screenshot("figure.png")
# exporter.screenshot("figure_hd.png", width=1920, height=1080)
#
# exporter.start_capture(width=800, height=600)
# for _ in range(90):
#     ... update entities ...
#     viz.flush()
#     exporter.capture_frame()
# exporter.finish_capture(video_path="orbit.mp4", fps=30)



## 5. Inline and standalone previews

`display_snapshot()` renders a static, serverless inline view; `open_snapshot()`
writes the self-contained HTML to a temp file and opens it in a browser.


In [6]:
viz = Visualizer(title="Export — preview")
viz.add(Sphere(Point(0, 0, 0), 2.0), opacity=0.4)

viz.display_snapshot()   # static inline view (serverless)
# viz.open_snapshot()    # standalone browser preview
print("previews covered")


previews covered


## 6. Animated export

`start_animation_recording()` returns a recording; capture the scene state per
frame, then embed it with `export_snapshot(animation=rec)` /
`export_figure(animation=rec)`. `AnimStyle` controls `fps`, `loop`,
`show_controls`, and `compress`.


In [7]:
viz = Visualizer(title="Export — animated")
point = viz.new(Point(3, 0, 0), color="#ff4444", label="orbit")
viz.flush()

recording = viz.start_animation_recording()
for frame in range(60):
    angle = frame * 0.1
    point.entity = Point(3 * math.cos(angle), 3 * math.sin(angle), 0)
    viz.flush()
    recording.capture_frame()

viz.export_snapshot(
    "_output/19_animated.html",
    overwrite=True,
    animation=recording,
    anim_style=AnimStyle(fps=30, loop=True, compress=True),
)


## 7. Keyboard shortcuts and camera notes

Exported HTML files (static, figure, animated) support `Ctrl+S` / `Cmd+S`
(download a PNG snapshot) and `r` (toggle camera auto-rotation). Exports apply
the full live-scene camera config (the default 2D view uses an orthographic
camera), and animated exports capture the camera per frame — so `set_camera()`
inside the loop is reflected in the exported animation.


## Visual Examples

The complete export set — HTML, figure, GLB, and animated HTML.


In [8]:
viz = Visualizer(title="Export — full set")
viz.add(Point(2, 0, 0), color="#ff4444", style=PointStyle(size=0.15), label="$P$")
viz.add(Sphere(Point(0, 0, 0), 2.0), style=SphereStyle(wireframe=True), opacity=0.3)
viz.flush()

viz.export_snapshot("_output/19_scene.html", overwrite=True)
viz.export_figure("_output/19_figure.html", style=FigureStyle(width=800, height=600),
                  overwrite=True)
viz.export_glb("_output/19_scene.glb", overwrite=True)

recording = viz.start_animation_recording()
for frame in range(30):
    angle = frame * 0.2
    # update a point, capture the camera each frame
    viz.flush()
    recording.capture_frame()

viz.export_snapshot("_output/19_animated.html", overwrite=True, animation=recording,
                    anim_style=AnimStyle(fps=30, loop=True))


## Summary

| Task | API |
|---|---|
| Standalone HTML | `export_snapshot(path)` |
| Embeddable figure | `export_figure(path, style=FigureStyle(...))` (or `path=None`) |
| glTF/GLB | `export_glb(path)` (no `export_gltf`) |
| PNG screenshot | `SceneExporter(viz).screenshot(path)` |
| MP4 video | `SceneExporter(viz).start_capture` / `capture_frame` / `finish_capture` |
| Inline preview | `display_snapshot()` |
| Standalone preview | `open_snapshot()` |
| Animated export | `start_animation_recording()` + `export_snapshot(animation=rec, anim_style=...)` |

**Next:** [20 — GA Entities](../20_ga_entities/).
